# Radar Financeiro — tabela inicial (5)

Gerar uma linha por MCI e execução em `SBX_A2A13E2.ANL_RADAR_FNC_CLI`. `FLAG_PARTICIPA_RADAR` recebe `'true'` quando as seis flags de elegibilidade são `'true'`. Um MCI identificado nos nichos Agro recebe `FLAG_AGRO = 'false'` e permanece na tabela com a participação desligada.

A qualificação usa os produtos `CD_PRD IN (6, 9)`. A moeda será filtrada na leitura dos movimentos do ciclo. A saída desta etapa tem os 15 campos do [DDL da tabela inicial](DDL_Radar_V6%20%281%29.ipynb).

## 1. Conexão

> **Conexão protegida. É proibido alterar, substituir ou excluir as células de conexão.**


Use os gerenciadores carregados abaixo para acessar as fontes do Radar. O `gerenciador_spark_v2.ipynb` fornece utilitários para a sessão Spark já iniciada, e o `gerenciador_db2_spark_v2.ipynb` fornece o conector para consultas ao DB2.


In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_local = GerenciadorLocal(
        nome_sessao="radar-todos-1",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_local.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        num_executors=6,
        executor_memory="4g",
        executor_cores=4,
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "2g",

            "spark.dynamicAllocation.enabled": "true",
            "spark.dynamicAllocation.minExecutors": "4",
            "spark.dynamicAllocation.initialExecutors": "6",
            "spark.dynamicAllocation.maxExecutors": "8",

            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",

            "spark.speculation": "false",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
            "spark.sql.adaptive.advisoryPartitionSizeInBytes": "64m",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "320",

            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

    print("Sessão Spark inicializada.")

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb


## 2. Parâmetros da execução

Preencha `dt_exea_manual` com uma data no formato `AAAA-MM-DD` para usá-la na execução. Com `None`, o notebook usa a variável de ambiente `HOJE`. A data escolhida é gravada na tabela como `DT_EXEA`. O parâmetro `quantidade_ciclos` define tanto os meses de renda quanto o período de análise. Os valores iniciais são `quantidade_ciclos = 1` e `particoes_jdbc = 4`.

As linhas `[RADAR_METRICA]` registram a etapa, o status e o tempo. As contagens aproveitam as materializações da base, do contexto e do resultado.


In [ ]:
%%spark
from contextlib import contextmanager, suppress
from datetime import date
from functools import reduce
import json
from time import perf_counter

from pyspark import StorageLevel
from pyspark.sql.window import Window

metricas_radar = {}


@contextmanager
def medir(etapa, tipo):
    inicio = perf_counter()
    dados = {"etapa": etapa, "tipo": tipo, "status": "ok"}
    print(f"[RADAR_INICIO] {etapa} ({tipo})")
    try:
        yield dados
    except (Exception, KeyboardInterrupt) as erro:
        dados["status"] = "interrompido" if isinstance(erro, KeyboardInterrupt) else "erro"
        dados["erro"] = type(erro).__name__
        raise
    finally:
        dados["segundos"] = round(perf_counter() - inicio, 3)
        metricas_radar[etapa] = dados
        print(f"[RADAR_METRICA] {json.dumps(dados, ensure_ascii=False)}")


def materializar(df, dados, anterior=None, flags=(), indicadores=()):
    df.persist(StorageLevel.MEMORY_AND_DISK)
    try:
        resultado = df.agg(
            F.count("*").alias("mcis"),
            *(F.count(F.when(F.col(flag) == "true", 1)).alias(flag) for flag in flags),
            *indicadores,
        ).first()
        dados.update(resultado.asDict())
        total_base = metricas_radar.get("Público Base", {}).get("mcis")
        if dados["etapa"] != "Público Base" and total_base is not None and dados["mcis"] != total_base:
            raise RuntimeError(f"{dados['etapa']} alterou a quantidade de MCIs: {total_base} para {dados['mcis']}.")
    except (Exception, KeyboardInterrupt):
        with suppress(Exception):
            df.unpersist()
        raise
    if anterior is not None:
        anterior.unpersist()
    return df


dt_exea_manual = None  # Exemplo: "2026-11-25".

with medir("Parâmetros da execução", "preparacao") as dados:
    tabela_inicial = None
    if dt_exea_manual is None:
        dt_exea = date.fromisoformat(str(obter_variavel_ambiente("HOJE"))[:10])
    else:
        dt_exea = date.fromisoformat(dt_exea_manual)
    quantidade_ciclos = 1
    particoes_jdbc = 4
    conector_db2 = criar_conector_db2_spark()
    dados.update(dt_exea=dt_exea.isoformat(), ciclos=quantidade_ciclos, particoes_jdbc=particoes_jdbc)


## 3. Entrada do MCI

A leitura por `CD_CLI` forma a base de clientes (`CD_CLI_REF`), fornece a última sincronização (`TS_INCL_TRAN_REF`) e identifica pessoa física (`FLAG_PESSOA_FISICA`). A consulta de nichos acrescenta `FLAG_AGRO` ao mesmo MCI.


### Filtro inicial pela data de sincronização — Público Base

Em `DB2GFP.TRAN_RLZD_INST_PCT`, selecionar os registros com `CD_CLI IS NOT NULL`, `CD_PRD IN (6, 9)`, `CD_EST_TRAN_INST = 0` e `DT_EXEA − 1 mês ≤ TS_INCL_TRAN < DT_EXEA`.

O DB2 resume os registros por partição (`NR_PTC`) e cliente (`CD_CLI`). O Spark reúne as partições e mantém uma linha por `CD_CLI_REF`.


In [ ]:
%%spark
with medir("Público Base", "execucao") as dados:
    tabela_inicial = None
    publico_por_particao = conector_db2.sql(f"""
        SELECT
            NR_PTC,
            CD_CLI,
            MAX(TS_INCL_TRAN) AS TS_MAX,
            MAX(CASE WHEN CD_TIP_PSS = 1 THEN 1 ELSE 0 END) AS TEM_PF,
            MIN(CPF_PF) AS CPF_MIN,
            MAX(CPF_PF) AS CPF_MAX,
            MIN(AGENCIA_BB) AS AG_MIN,
            MAX(AGENCIA_BB) AS AG_MAX,
            MIN(CONTA_BB) AS CC_MIN,
            MAX(CONTA_BB) AS CC_MAX
        FROM (
            SELECT
                NR_PTC,
                CD_CLI,
                TS_INCL_TRAN,
                CD_TIP_PSS,
                CASE WHEN CD_TIP_PSS = 1 THEN NR_CPF_CNPJ_TITR END AS CPF_PF,
                CASE WHEN CD_TIP_PSS = 1 AND NR_MCA_PCT_OPB = 999999999
                          AND NR_AG_TITR IS NOT NULL AND TRIM(CD_CT_TITR) <> ''
                     THEN NR_AG_TITR
                END AS AGENCIA_BB,
                CASE WHEN CD_TIP_PSS = 1 AND NR_MCA_PCT_OPB = 999999999
                          AND NR_AG_TITR IS NOT NULL AND TRIM(CD_CT_TITR) <> ''
                     THEN TRIM(CD_CT_TITR)
                END AS CONTA_BB
            FROM DB2GFP.TRAN_RLZD_INST_PCT
            WHERE CD_CLI IS NOT NULL
              AND CD_PRD IN (6, 9)
              AND CD_EST_TRAN_INST = 0
              AND TS_INCL_TRAN >= TIMESTAMP(DATE('{dt_exea}') - 1 MONTH)
              AND TS_INCL_TRAN < TIMESTAMP(DATE('{dt_exea}'))
        ) MOV
        GROUP BY NR_PTC, CD_CLI
    """,
        partition_column="NR_PTC",
        lower_bound=1,
        upper_bound=101,
        num_partitions=particoes_jdbc,
    )

    publico_agregado = materializar(
        publico_por_particao
        .withColumnRenamed("CD_CLI", "CD_CLI_REF")
        .groupBy("CD_CLI_REF")
        .agg(
            F.max("TS_MAX").alias("TS_ULTIMA_INCLUSAO"),
            F.max("TEM_PF").alias("TEM_PF"),
            F.min("CPF_MIN").alias("CPF_MIN"),
            F.max("CPF_MAX").alias("CPF_MAX"),
            F.min("AG_MIN").alias("AG_MIN"),
            F.max("AG_MAX").alias("AG_MAX"),
            F.min("CC_MIN").alias("CC_MIN"),
            F.max("CC_MAX").alias("CC_MAX"),
        ),
        dados,
    )
    publico_base = publico_agregado


### Adicionar a última sincronização por MCI

Na mesma leitura, calcular `MAX(TS_INCL_TRAN)` por `NR_PTC` e `CD_CLI`. Ao consolidar as partições, guardar o maior desses valores em `TS_INCL_TRAN_REF` para cada `CD_CLI_REF`.


In [ ]:
%%spark
with medir("Última sincronização por MCI", "preparacao") as dados:
    publico_base = publico_agregado.withColumnRenamed("TS_ULTIMA_INCLUSAO", "TS_INCL_TRAN_REF")


### Avaliar o critério Agro por MCI

Em `DB2SMI.SUB_NHMC_CLI`, consultar o nicho principal do cliente (`IN_NHMC_PPL_CLI = 'S'`) e os códigos `NR_SEQL_NHMC IN (31, 45, 59, 60, 61, 76, 88, 125, 133, 165, 210)`. Cruzar `CD_CLI` com `CD_CLI_REF`.

Com correspondência, gravar `FLAG_AGRO = 'false'` e excluir o MCI da participação. Sem correspondência, gravar `FLAG_AGRO = 'true'` e considerar as demais flags.

In [ ]:
%%spark
with medir("Agro por MCI", "preparacao") as dados:
    agro_por_mci = conector_db2.sql("""
        SELECT DISTINCT
            CAST(agro.CD_CLI AS INTEGER) AS CD_CLI_REF,
            'false' AS FLAG_AGRO
        FROM DB2SMI.SUB_NHMC_CLI agro
        WHERE agro.CD_CLI IS NOT NULL
          AND agro.IN_NHMC_PPL_CLI = 'S'
          AND agro.NR_SEQL_NHMC IN (31, 45, 59, 60, 61, 76, 88, 125, 133, 165, 210)
    """)

    publico_base = (
        publico_base.drop("FLAG_AGRO")
        .join(agro_por_mci, "CD_CLI_REF", "left")
        .fillna("true", subset=["FLAG_AGRO"])
    )

### Identificar pessoa física

Para cada `CD_CLI_REF`, usar a leitura de sincronização da seção 3. Se ao menos um registro da janela tiver `CD_TIP_PSS = 1`, gravar `FLAG_PESSOA_FISICA = 'true'`. Caso contrário, gravar `'false'`.


In [ ]:
%%spark
with medir("Pessoa física", "preparacao") as dados:
    publico_base = (
        publico_base
        .withColumn("FLAG_PESSOA_FISICA", (F.col("TEM_PF") == 1).cast("string"))
        .drop("TEM_PF")
    )


## 4. Identificação e contexto

Cada regra enriquece a linha de `CD_CLI_REF`. A conta identificada por agência e número serve à consulta de `DD_INC_MM_CLC_BLC_REF`. O CPF identificado serve à busca de `VL_REN_PRES_REF`. O macroperfil é consultado pelo próprio MCI e guardado em `CD_MAC_PRFL_CLI_REF`. As flags das regras ainda não avaliadas ficam nulas.


### Identificar conta corrente BB única

Para o MCI com `FLAG_PESSOA_FISICA = 'true'`, usar os registros de `CD_TIP_PSS = 1`, marca BB (`NR_MCA_PCT_OPB = 999999999`) e produtos `CD_PRD IN (6, 9)`. Considerar a agência preenchida (`NR_AG_TITR`) e a conta preenchida após remover os espaços nas pontas (`CD_CT_TITR`).

Há um único par quando os extremos de agência e conta coincidem. O mesmo par nos dois produtos conta uma vez. Gravar `FLAG_CONTA_CORRENTE_BB_UNICA = 'true'` quando a agência for não negativa e a conta contiver apenas dígitos e puder ser convertida. As chaves resultantes `CD_UOR_CC` e `NR_CC` servem à consulta do dia do ciclo.


In [ ]:
%%spark
with medir("Conta corrente BB única", "preparacao") as dados:
    conta_unica = F.expr("AG_MIN = AG_MAX AND CC_MIN = CC_MAX")
    contexto = (
        publico_base
        .withColumn("CD_UOR_CC", F.when(
            conta_unica & (F.col("AG_MIN") >= 0), F.col("AG_MIN").cast("int")
        ))
        .withColumn("NR_CC", F.when(
            conta_unica & F.col("CC_MIN").rlike("^[0-9]+$"),
            F.expr("TRY_CAST(COALESCE(NULLIF(REGEXP_REPLACE(CC_MIN, '^0+', ''), ''), '0') AS DECIMAL(11,0))")
        ))
        .withColumn("FLAG_CONTA_CORRENTE_BB_UNICA", F.expr("""
            CASE WHEN FLAG_PESSOA_FISICA = 'true'
                 THEN CAST(CD_UOR_CC IS NOT NULL AND NR_CC IS NOT NULL AS STRING)
            END
        """))
        .drop("AG_MIN", "AG_MAX", "CC_MIN", "CC_MAX")
    )


### Identificar CPF único

Quando `FLAG_CONTA_CORRENTE_BB_UNICA = 'true'`, comparar o menor e o maior `NR_CPF_CNPJ_TITR` preenchido nos registros com `CD_TIP_PSS = 1` daquele MCI. Se existir um valor e os extremos coincidirem, gravar `FLAG_CPF_UNICO = 'true'` e usar o número em `CPF_UNICO` para consultar a renda. Com zero ou mais de um CPF distinto, a flag recebe `'false'` e `CPF_UNICO` fica nulo.

A materialização desta etapa já inclui `FLAG_AGRO`, `FLAG_PESSOA_FISICA`, `FLAG_CONTA_CORRENTE_BB_UNICA` e `FLAG_CPF_UNICO`. A estimativa de CPFs distintos dimensiona a busca de renda.


In [ ]:
%%spark
with medir("Identificação inicial e CPF", "execucao") as dados:
    contexto = materializar(
        contexto
        .withColumn("FLAG_CPF_UNICO", F.expr("""
            CASE WHEN FLAG_CONTA_CORRENTE_BB_UNICA = 'true'
                 THEN CAST(COALESCE(CPF_MIN = CPF_MAX, FALSE) AS STRING)
            END
        """))
        .withColumn("CPF_UNICO", F.when(
            F.col("FLAG_CPF_UNICO") == "true", F.col("CPF_MIN").cast("decimal(14,0)")
        ))
        .drop("CPF_MIN", "CPF_MAX"),
        dados,
        anterior=publico_agregado,
        flags=("FLAG_AGRO", "FLAG_PESSOA_FISICA", "FLAG_CONTA_CORRENTE_BB_UNICA", "FLAG_CPF_UNICO"),
        indicadores=(F.approx_count_distinct("CPF_UNICO").alias("cpfs_distintos_estimados"),),
    )


### Consultar renda presumida

Para os MCIs com `FLAG_CPF_UNICO = 'true'`, cruzar `CPF_UNICO` com `DB2DFE.REN_AVLD_PF.NR_CPF_BASE_SRF`. Selecionar a renda `VL_REN` do registro mais recente por `DT_INCL_REN_AVLD`.

Guardar `VL_REN_PRES_REF = VL_REN × quantidade_ciclos`. Um valor preenchido, inclusive zero, recebe `FLAG_RENDA_PRESUMIDA_DISPONIVEL = 'true'`. O CPF temporário é descartado após a consulta.


In [ ]:
%%spark
with medir("Renda presumida", "execucao") as dados:
    cpfs_elegiveis = (
        contexto.where("FLAG_CPF_UNICO = 'true'")
        .select("CPF_UNICO")
        .distinct()
    )
    renda = (
        spark.table("DB2DFE.REN_AVLD_PF")
        .select(
            F.col("NR_CPF_BASE_SRF").cast("decimal(14,0)").alias("CPF_UNICO"),
            "DT_INCL_REN_AVLD", "VL_REN",
        )
        .join(cpfs_elegiveis, "CPF_UNICO", "left_semi")
        .withColumn("RN", F.row_number().over(
            Window.partitionBy("CPF_UNICO").orderBy(F.col("DT_INCL_REN_AVLD").desc_nulls_last())
        ))
        .where("RN = 1")
        .select(
            "CPF_UNICO",
            (F.col("VL_REN") * quantidade_ciclos).cast("decimal(17,2)").alias("VL_REN_PRES_REF"),
        )
    )

    contexto = materializar(
        contexto.join(renda, "CPF_UNICO", "left")
        .withColumn("FLAG_RENDA_PRESUMIDA_DISPONIVEL", F.expr("""
            CASE WHEN FLAG_CPF_UNICO = 'true'
                 THEN CAST(VL_REN_PRES_REF IS NOT NULL AS STRING)
            END
        """))
        .drop("CPF_UNICO"),
        dados,
        anterior=contexto,
        flags=("FLAG_RENDA_PRESUMIDA_DISPONIVEL",),
    )


### Consultar macroperfil

Para os MCIs com `FLAG_RENDA_PRESUMIDA_DISPONIVEL = 'true'`, localizar `CD_CLI` em `DB2D1D.DVS_GRDR_FNCO_PF`. A consulta ao DB2 limita a leitura a `DT_EXEA − 6 meses ≤ DT_REF ≤ DT_EXEA` e seleciona a última data disponível de cada MCI.

Se nessa data existir um único código preenchido, guardar o valor de `CD_MAC_PRFL_CLI` em `CD_MAC_PRFL_CLI_REF` e marcar `FLAG_MACROPERFIL_VALIDO = 'true'`. Repetições do mesmo código valem como uma identificação. Sem código ou com códigos diferentes na última data, a flag recebe `'false'` e o atributo fica nulo. Quando a regra de renda não foi aprovada, a flag de macroperfil permanece nula.

In [ ]:
%%spark
def ler_db2_por_faixas(chaves, coluna_spark, coluna_db2, consulta, esquema, dados):
    minimo, maximo = chaves.agg(F.min(coluna_spark), F.max(coluna_spark)).first()
    if minimo is None:
        dados["faixas_jdbc"] = 0
        return spark.createDataFrame([], esquema)

    minimo, maximo = int(minimo), int(maximo)
    largura = (maximo - minimo + particoes_jdbc) // particoes_jdbc
    inicios = range(minimo, maximo + 1, largura)
    dados["faixas_jdbc"] = len(inicios)
    leituras = [
        conector_db2.sql(
            consulta.format(faixa=f"{coluna_db2} BETWEEN {inicio} AND {min(maximo, inicio + largura - 1)}"),
            query_timeout=3600,
        )
        for inicio in inicios
    ]
    return reduce(lambda primeira, segunda: primeira.unionByName(segunda), leituras)


with medir("Macroperfil", "execucao") as dados:
    mcis_elegiveis = contexto.where("FLAG_RENDA_PRESUMIDA_DISPONIVEL = 'true'").select("CD_CLI_REF")
    perfil = ler_db2_por_faixas(
        mcis_elegiveis, "CD_CLI_REF", "CD_CLI",
        f"""
        SELECT CAST(CD_CLI AS INTEGER) AS CD_CLI_REF,
               CASE WHEN MIN(CD_MAC_PRFL_CLI) = MAX(CD_MAC_PRFL_CLI)
                    THEN CAST(MIN(CD_MAC_PRFL_CLI) AS INTEGER)
               END AS CD_MAC_PRFL_CLI_REF
        FROM (
            SELECT CD_CLI, CD_MAC_PRFL_CLI,
                   DENSE_RANK() OVER (PARTITION BY CD_CLI ORDER BY DT_REF DESC) AS ORDEM_DATA
            FROM DB2D1D.DVS_GRDR_FNCO_PF
            WHERE DT_REF >= DATE('{dt_exea.isoformat()}') - 6 MONTHS
              AND DT_REF <= DATE('{dt_exea.isoformat()}') AND ({{faixa}})
        ) P
        WHERE ORDEM_DATA = 1
        GROUP BY CD_CLI
        """,
        "CD_CLI_REF INT, CD_MAC_PRFL_CLI_REF INT", dados,
    ).join(mcis_elegiveis, "CD_CLI_REF", "left_semi")

    contexto = materializar(
        contexto.join(perfil, "CD_CLI_REF", "left")
        .withColumn("FLAG_MACROPERFIL_VALIDO", F.expr("""
            CASE WHEN FLAG_RENDA_PRESUMIDA_DISPONIVEL = 'true'
                 THEN CAST(CD_MAC_PRFL_CLI_REF IS NOT NULL AS STRING)
            END
        """)),
        dados,
        anterior=contexto,
        flags=("FLAG_MACROPERFIL_VALIDO",),
    )


### Consultar o dia do ciclo financeiro

Com `FLAG_MACROPERFIL_VALIDO = 'true'`, consultar `DB2GFP.CT_GRDR_FNCO` pela agência `CD_UOR_CC` e pela conta `NR_CC`. Para cada par, selecionar o registro mais recente por `TS_ULT_EXEA_PSQ`.

Guardar o valor cadastral de `DD_INC_MM_CLC_BLC` em `DD_INC_MM_CLC_BLC_REF`. Se faltar o dia, essa coluna fica nula. As chaves de agência e conta são descartadas após a consulta.


In [ ]:
%%spark
with medir("Dia do ciclo", "preparacao") as dados:
    contas_elegiveis = (
        contexto.where("FLAG_MACROPERFIL_VALIDO = 'true'")
        .select("CD_UOR_CC", "NR_CC")
        .distinct()
    )

    ciclo = ler_db2_por_faixas(
        contas_elegiveis, "CD_UOR_CC", "CD_UOR_CC",
        """
        SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC, TS_ULT_EXEA_PSQ
        FROM DB2GFP.CT_GRDR_FNCO
        WHERE {faixa}
        """,
        "CD_UOR_CC INT, NR_CC DECIMAL(11,0), "
        "DD_INC_MM_CLC_BLC SMALLINT, TS_ULT_EXEA_PSQ TIMESTAMP",
        dados,
    )

    ciclo = (
        ciclo.withColumn(
            "RN",
            F.row_number().over(
                Window.partitionBy("CD_UOR_CC", "NR_CC")
                .orderBy(F.col("TS_ULT_EXEA_PSQ").desc_nulls_last())
            ),
        )
        .where("RN = 1")
        .selectExpr(
            "CAST(CD_UOR_CC AS INT) AS CD_UOR_CC",
            "CAST(NR_CC AS DECIMAL(11,0)) AS NR_CC",
            "CAST(DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC_REF",
        )
        .join(contas_elegiveis, ["CD_UOR_CC", "NR_CC"], "left_semi")
    )

    contexto_ciclo = (
        contexto.join(ciclo, ["CD_UOR_CC", "NR_CC"], "left")
        .withColumn(
            "DD_INC_MM_CLC_BLC_REF",
            F.when(
                F.col("FLAG_MACROPERFIL_VALIDO") == "true",
                F.col("DD_INC_MM_CLC_BLC_REF"),
            ),
        )
        .drop("CD_UOR_CC", "NR_CC")
    )


## 5. Período de análise

O código calcula o período para o MCI quando todas estas flags valem `'true'` — `FLAG_AGRO`, `FLAG_PESSOA_FISICA`, `FLAG_CONTA_CORRENTE_BB_UNICA`, `FLAG_CPF_UNICO`, `FLAG_RENDA_PRESUMIDA_DISPONIVEL` e `FLAG_MACROPERFIL_VALIDO`.

Usar a data de `TS_INCL_TRAN_REF` e o dia cadastrado em `DD_INC_MM_CLC_BLC_REF` para localizar o início do ciclo aberto. `DT_FIM_REF` é a véspera desse início. `DT_INI_REF` é o início do primeiro ciclo fechado incluído, conforme `quantidade_ciclos`. O cálculo usa dia 1 quando falta o dia cadastrado e usa o último dia do mês quando ele é mais curto.


In [ ]:
%%spark
with medir("Período de análise", "preparacao") as dados:
    condicoes_participacao = F.expr("""
        FLAG_AGRO = 'true'
        AND FLAG_PESSOA_FISICA = 'true'
        AND FLAG_CONTA_CORRENTE_BB_UNICA = 'true'
        AND FLAG_CPF_UNICO = 'true'
        AND FLAG_RENDA_PRESUMIDA_DISPONIVEL = 'true'
        AND FLAG_MACROPERFIL_VALIDO = 'true'
    """)


    def inicio_ciclo_no_mes(mes, dia):
        return F.date_add(mes, F.least(dia, F.dayofmonth(F.last_day(mes))) - 1)


    data_sincronizacao = F.to_date("TS_INCL_TRAN_REF")
    mes_sincronizacao = F.trunc(data_sincronizacao, "month")
    dia_ciclo = F.coalesce(F.col("DD_INC_MM_CLC_BLC_REF").cast("int"), F.lit(1))
    inicio_no_mes = inicio_ciclo_no_mes(mes_sincronizacao, dia_ciclo)
    inicio_ciclo_aberto = F.when(data_sincronizacao >= inicio_no_mes, inicio_no_mes).otherwise(
        inicio_ciclo_no_mes(F.add_months(mes_sincronizacao, -1), dia_ciclo)
    )
    mes_inicio_periodo = F.add_months(F.trunc(inicio_ciclo_aberto, "month"), -quantidade_ciclos)

    contexto_ciclo = (
        contexto_ciclo
        .withColumn("DT_INI_REF", F.when(
            condicoes_participacao, inicio_ciclo_no_mes(mes_inicio_periodo, dia_ciclo)
        ))
        .withColumn("DT_FIM_REF", F.when(
            condicoes_participacao, F.date_sub(inicio_ciclo_aberto, 1)
        ))
    )


## 6. Montar os 15 atributos de qualificação

A base completa `tabela_inicial` registra `FLAG_PARTICIPA_RADAR = 'true'` para os MCIs que atendem às seis condições da seção 5 e `'false'` para os demais. A ordem de obtenção segue as dependências de cada busca. A projeção da célula de montagem segue a ordem física do DDL existente.

| Quando o atributo fica disponível | Atributos da principal |
|---|---|
| Execução e formação da base por cliente | `DT_EXEA`, `CD_CLI_REF`, `TS_INCL_TRAN_REF` |
| Indicação Agro e identificação da pessoa física | `FLAG_AGRO`, `FLAG_PESSOA_FISICA` |
| Conta BB única e CPF único | `FLAG_CONTA_CORRENTE_BB_UNICA`, `FLAG_CPF_UNICO` |
| Renda e macroperfil | `FLAG_RENDA_PRESUMIDA_DISPONIVEL`, `VL_REN_PRES_REF`, `FLAG_MACROPERFIL_VALIDO`, `CD_MAC_PRFL_CLI_REF` |
| Consulta do dia do ciclo | `DD_INC_MM_CLC_BLC_REF` |
| Período e participação | `DT_INI_REF`, `DT_FIM_REF`, `FLAG_PARTICIPA_RADAR` |

In [ ]:
%%spark
with medir("Ciclo, período e tabela inicial", "execucao") as dados:
    tabela_inicial = None
    tabela_inicial = materializar(
        contexto_ciclo
        .withColumn("DT_EXEA", F.lit(dt_exea))
        .withColumn("FLAG_PARTICIPA_RADAR", F.coalesce(condicoes_participacao, F.lit(False)).cast("string"))
        .select(
            "DT_EXEA", "CD_CLI_REF", "TS_INCL_TRAN_REF", "FLAG_AGRO",
            "FLAG_PESSOA_FISICA", "FLAG_CONTA_CORRENTE_BB_UNICA", "DD_INC_MM_CLC_BLC_REF",
            "FLAG_CPF_UNICO", "FLAG_RENDA_PRESUMIDA_DISPONIVEL", "VL_REN_PRES_REF",
            "FLAG_MACROPERFIL_VALIDO", "CD_MAC_PRFL_CLI_REF",
            "DT_INI_REF", "DT_FIM_REF", "FLAG_PARTICIPA_RADAR",
        ),
        dados,
        anterior=contexto,
        flags=("FLAG_PARTICIPA_RADAR",),
        indicadores=(
            F.count("DD_INC_MM_CLC_BLC_REF").alias("mcis_com_dia_cadastrado"),
            F.count("DT_INI_REF").alias("mcis_com_periodo"),
        ),
    )

print(f"[RADAR_RESUMO] {json.dumps(list(metricas_radar.values()), ensure_ascii=False)}")


## 7. Gravação da tabela inicial

Gravar o resultado completo em `SBX_A2A13E2.ANL_RADAR_FNC_CLI`, criada pelo notebook de DDL. Esta célula substitui os dados existentes pelo resultado da execução atual.


In [ ]:
%%spark
with medir("Gravação da tabela inicial", "execucao") as dados:
    if tabela_inicial is None:
        raise RuntimeError("Execute a montagem da tabela inicial antes da gravação.")
    dados["mcis"] = metricas_radar["Ciclo, período e tabela inicial"]["mcis"]
    tabela_inicial.write.insertInto("SBX_A2A13E2.ANL_RADAR_FNC_CLI", overwrite=True)


## 8. Conferir a tabela gravada

Ler `SBX_A2A13E2.ANL_RADAR_FNC_CLI` e reunir em uma agregação por `DT_EXEA` a quantidade de linhas, os MCIs distintos (`CD_CLI_REF`), a distribuição das flags e o preenchimento do período (`DT_INI_REF` e `DT_FIM_REF`). O relatório usa o resultado lido da tabela.

In [ ]:
%%spark
from pyspark.sql import functions as F

nome_tabela = "SBX_A2A13E2.ANL_RADAR_FNC_CLI"

if not spark.catalog.tableExists(nome_tabela):
    raise RuntimeError(f"Tabela não encontrada: {nome_tabela}")

base_salva = spark.table(nome_tabela)

print(f"TABELA: {nome_tabela}")
print("Fonte: dados gravados, não o DataFrame tabela_inicial em memória.")
print("\nCAMPOS E SIGNIFICADOS:")

for campo in base_salva.schema.fields:
    descricao = campo.metadata.get("comment", "Sem comentário no catálogo")

    if campo.name == "FLAG_AGRO":
        descricao = "True quando o MCI está fora dos nichos Agro consultados. False quando é identificado como Agro."
    elif campo.name == "FLAG_MACROPERFIL_VALIDO":
        descricao = "Indica se há um único código preenchido na data de perfil mais recente nos últimos seis meses"

    print(f"  {campo.name} [{campo.dataType.simpleString()}]: {descricao}")

flags = [
    (
        "FLAG_AGRO",
        "Identificação Agro",
        "não encontrado nos 11 nichos Agro consultados",
        "encontrado nos nichos Agro consultados",
        "sem classificação; não esperado após a regra de preenchimento",
    ),
    (
        "FLAG_PESSOA_FISICA",
        "Pessoa física",
        "há registro de pessoa física na janela consultada",
        "não há registro de pessoa física na janela consultada",
        "sem valor para a classificação de pessoa física",
    ),
    (
        "FLAG_CONTA_CORRENTE_BB_UNICA",
        "Conta corrente BB única",
        "uma única conta BB foi identificada em formato válido",
        "a conta não atende à regra de unicidade e formato válido",
        "não avaliada porque a condição de pessoa física não foi atendida",
    ),
    (
        "FLAG_CPF_UNICO",
        "CPF único",
        "foi identificado exatamente um CPF preenchido",
        "não foi identificado exatamente um CPF preenchido",
        "não avaliado porque a condição de conta BB não foi atendida",
    ),
    (
        "FLAG_RENDA_PRESUMIDA_DISPONIVEL",
        "Renda presumida disponível",
        "foi encontrado um valor de renda; zero também conta como disponível",
        "a consulta foi feita, mas não retornou valor de renda",
        "não avaliada porque a condição de CPF único não foi atendida",
    ),
    (
        "FLAG_MACROPERFIL_VALIDO",
        "Macroperfil válido",
        "um código de macroperfil identificado na data mais recente da janela",
        "código ausente ou conflitante na data mais recente da janela",
        "não avaliado porque a condição de renda disponível não foi atendida",
    ),
    (
        "FLAG_PARTICIPA_RADAR",
        "Participa do Radar",
        "todas as seis condições de participação foram atendidas",
        "uma ou mais condições de participação não foram atendidas",
        "sem decisão final; não esperado na montagem atual",
    ),
]

indicadores = [
    F.count("*").alias("total"),
    F.count("CD_CLI_REF").alias("mcis_preenchidos"),
    F.countDistinct("CD_CLI_REF").alias("mcis_unicos"),
    F.count("TS_INCL_TRAN_REF").alias("com_sincronizacao"),
    F.count("DD_INC_MM_CLC_BLC_REF").alias("com_dia_ciclo"),
    F.count("VL_REN_PRES_REF").alias("com_renda"),
    F.count(
        F.when(F.col("VL_REN_PRES_REF") == 0, 1)
    ).alias("renda_zero"),
    F.count("CD_MAC_PRFL_CLI_REF").alias("com_macroperfil"),
    F.count(
        F.when(
            F.col("DT_INI_REF").isNotNull()
            & F.col("DT_FIM_REF").isNotNull(),
            1,
        )
    ).alias("com_periodo"),
    F.min("DT_INI_REF").alias("menor_inicio"),
    F.max("DT_FIM_REF").alias("maior_fim"),
]

for coluna, _, _, _, _ in flags:
    valor = F.col(coluna)
    indicadores.extend([
        F.count(F.when(valor == "true", 1)).alias(f"{coluna}_true"),
        F.count(F.when(valor == "false", 1)).alias(f"{coluna}_false"),
        F.count(F.when(valor.isNull(), 1)).alias(f"{coluna}_nulos"),
        F.count(
            F.when(
                valor.isNotNull() & ~valor.isin("true", "false"),
                1,
            )
        ).alias(f"{coluna}_outros"),
    ])

resumos = (
    base_salva
    .groupBy("DT_EXEA")
    .agg(*indicadores)
    .orderBy(F.col("DT_EXEA").desc_nulls_last())
    .collect()
)

def formatar_numero(valor):
    return f"{valor:,}".replace(",", ".")

if not resumos:
    print("\nATENÇÃO: a tabela existe, mas está vazia.")

for resumo in resumos:
    r = resumo.asDict()
    total = r["total"]

    print("\n" + "=" * 80)
    print(f"DATA DA EXECUÇÃO GRAVADA: {r['DT_EXEA']}")
    print(f"Total de registros: {formatar_numero(total)}")
    print(f"MCIs distintos: {formatar_numero(r['mcis_unicos'])}")
    print(
        "Registros sem MCI: "
        f"{formatar_numero(total - r['mcis_preenchidos'])}"
    )
    print(
        "Registros excedentes por MCI repetido: "
        f"{formatar_numero(r['mcis_preenchidos'] - r['mcis_unicos'])}"
    )

    print("\nCONTAGEM DAS FLAGS:")
    print("true = condição atendida, conforme a descrição da flag.")
    print("false = condição avaliada, mas não atendida.")
    print("nulo = condição sem decisão; pode não ter sido avaliada.")
    print("Percentuais de true calculados sobre o total de registros.")

    for coluna, nome, texto_true, texto_false, texto_nulo in flags:
        quantidade_true = r[f"{coluna}_true"]
        percentual_true = 100 * quantidade_true / total if total else 0

        print(f"\n{coluna} — {nome}")
        print(
            f"  Aprovados (true): {formatar_numero(quantidade_true)} "
            f"({percentual_true:.2f}%) — {texto_true}."
        )
        print(
            f"  Não aprovados (false): "
            f"{formatar_numero(r[f'{coluna}_false'])} — {texto_false}."
        )
        print(
            f"  Sem resultado (nulos): "
            f"{formatar_numero(r[f'{coluna}_nulos'])} — {texto_nulo}."
        )
        print(
            f"  Valores fora do padrão: "
            f"{formatar_numero(r[f'{coluna}_outros'])} — "
            "valor diferente de true, false ou nulo."
        )

    print("\nPREENCHIMENTO DOS DEMAIS CAMPOS:")
    print(
        f"Última sincronização preenchida: "
        f"{formatar_numero(r['com_sincronizacao'])}"
    )
    print(
        f"Dia de ciclo cadastrado: "
        f"{formatar_numero(r['com_dia_ciclo'])}"
    )
    print(
        f"Sem dia de ciclo cadastrado: "
        f"{formatar_numero(total - r['com_dia_ciclo'])}; "
        "o cálculo usa dia 1 quando aplicável."
    )
    print(
        f"Renda presumida preenchida: "
        f"{formatar_numero(r['com_renda'])}"
    )
    print(
        f"Renda igual a zero: {formatar_numero(r['renda_zero'])}; "
        "zero conta como renda disponível."
    )
    print(
        f"Código de macroperfil preenchido: "
        f"{formatar_numero(r['com_macroperfil'])}"
    )
    print(
        f"Período com início e fim preenchidos: "
        f"{formatar_numero(r['com_periodo'])}"
    )
    print(f"Menor início de período: {r['menor_inicio']}")
    print(f"Maior fim de período: {r['maior_fim']}")